# CVPR experiments: "align distributions, not positions" across representations

One notebook, everything in `masters_texture/cvpr_experiments/`. Every section is **resumable**: finished pairs are skipped, so a dead session costs nothing. Turn sections on and off with the `RUN_*` switches in the config cell.

| Section | Question it answers | Output |
|---|---|---|
| **A. DINO + SAM, feature level** | Does positional blending push tokens off the space of real tokens, and does matching avoid that? Two non-circular measures: distance to the nearest *real* token, and how much variance survives. | `feature_level/*.csv`, `figs/feature_level_*.png` |
| **B. DiffMorpher** | The modern diffusion morphing baseline, run with the official code. | `renders/diffmorpher/` |
| **C. CNT along the whole path** | KID/FID at 9 values of t, not only t = 0.5. Maybe another t looks even better. | `renders/cnt_*/` |
| **D. CNT + LoRA** | Can *fine-tuning* the decoder make positional blending as good as soft OT? Either answer helps the paper. | `renders/cnt_lora_linear/`, `figs/lora_training.png` |
| **E. DINO image level** | Interpolate DINOv2 tokens and decode them to images with the RAE decoder, so DINO gets real KID numbers. | `renders/rae_*/` |
| **F. Scores and graphs** | KID/FID vs t for every method, Delta-KID with paired-jackknife CIs, strips. Rerun any time. | `scores/*.csv`, `figs/*.png` |

**Methods everywhere:** positional (`field_linear`), target mean, Gaussian (moment matching), soft OT (the paper's), exact OT. Same 120 pairs, same 128 px endpoints, same KID protocol (Inception, frozen 2,819-image DTD reference) as the paper.

### How to run
1. **Session 1** (A100, about 1.5 h): run **config**, then **install**, then **Runtime → Restart session**. Run **config** again, then everything from **setup** down. That covers A, C, D, E and F. DiffMorpher is off by default.
2. **Sessions 2 and on** (about 100 min each): in **config** set `RUN_DIFFMORPHER = True` and the other `RUN_*` to `False`, except `RUN_SCORE = True`. Run config, install, restart, config, setup, B1, B2, F. Each session adds about 40 DiffMorpher pairs, so 120 pairs take about 3 sessions. F scores whatever exists, and DiffMorpher is always compared on the pairs it has.

### Before trusting any number
- **D:** the printed sanity line `|differentiable decode - package field_linear|` must be ~0.
- **E:** the RAE sanity image: the t = 0 frame should look like A.
- **F:** all CNT rules must have the same KID at t = 0, since every rule starts from the same reconstruction of A.
- **FID** uses only 120 images, so it is biased upward. Compare within a column only, and treat KID as the main measure.

In [ ]:
# ==== Config: run this cell first, and again after every restart ======================================================
import os
MOCK = os.environ.get("CVPR_MOCK") == "1"          # local CPU dry run with fake models (leave False on Colab)
DRIVE = "/content/drive/MyDrive"
MT = os.environ.get("CVPR_MT", f"{DRIVE}/masters_texture")      # everything is read from / written to here
OUT = f"{MT}/cvpr_experiments"                # all outputs: renders/, feats/, scores/, figs/, feature_level/, lora/
CNT_CKPTS = f"{MT}/cnt_ckpts"                     # finetuned-model256.ckpt (already on your Drive)
DTD_ROOT = f"{MT}/dtd/images"                     # DTD images/<category>/*.jpg (downloaded here if missing)
PACKAGE_ZIP = f"{DRIVE}/Labeled_Texton_Matching/textureMatcher_code_snapshot.zip"   # searched on Drive if not here

# ---- what to run in this session (everything is resumable: finished pairs are skipped) ----
RUN_FEATS       = True    # A. DINO + SAM feature-level tests (no images)
RUN_DIFFMORPHER = False   # B. DiffMorpher baseline (slow: ~2-3 min/pair; run in its own session(s))
RUN_CNT_SWEEP   = True    # C. CNT renders along the whole path, for the KID/FID-vs-t graph
RUN_LORA        = True    # D. LoRA-fine-tuned CNT decoder: can training make positional blending match soft OT?
RUN_RAE         = True    # E. DINO image-level: interpolate DINOv2 tokens and decode with the RAE decoder
RUN_SCORE       = True    # F. KID/FID along the path for everything rendered so far + all graphs

N_PAIRS = 120                                     # the paper's 120-pair evaluation subset
T_GRID = [0.0, 0.125, 0.25, 0.375, 0.5, 0.625, 0.75, 0.875, 1.0]
RULES = ["field_linear", "target_mean", "gaussian", "soft_ot", "soft_ot_sharp", "exact_ot"]   # positional + the matching family

# A / E: token spaces (DINO via RAE, SAM2)
RAE_REPO = "nyu-visionx/RAE-dinov2-wReg-base-ViTXL-n08"    # DINOv2-B (registers) encoder + ViT-XL pixel decoder, 224 px in
EPS_REL = 0.05            # soft-OT epsilon for token spaces = EPS_REL x median squared token distance
EPS_REL_SHARP = 0.005     # "soft_ot_sharp": 10x sharper (A showed eps=0.05 is near-uniform in 768-d DINO)
CNT_EPS_SHARP = 0.005     # CNT "soft_ot_sharp": the paper's sharp setting (raw units; paper default is 0.05)
N_EXACT_TOK = 1024        # exact OT: Hungarian on up to this many tokens, rest by nearest sampled token (as in the paper)
GAUSS_RIDGE_REL = 0.05    # Gaussian map ridge = this x mean token variance (DINO has 256 tokens in 768-d)
N_BANK = 300              # reference images (not in any eval pair) whose tokens define "what real tokens look like"
BANK_TOKENS_PER_IMG = 256

# B: DiffMorpher (official code, isolated Python 3.11 env, SD 2.1-base)
DM_N_PAIRS = 120
DM_TIME_BUDGET_MIN = 100  # stops cleanly after this many minutes; rerun the cell in a new session to continue
DM_LORA_STEPS = 200       # DiffMorpher default
DM_COMMIT = "111627b4188fc9726b85c24647e3826cca0b448f"
SD_DIR = f"{MT}/cache/sd21_base"
SD_REPOS = ["stabilityai/stable-diffusion-2-1-base", "Manojb/stable-diffusion-2-1-base",
            "RedbeardNZ/stable-diffusion-2-1-base"]   # official repo was pulled from HF; mirrors (safetensors only)

# D: LoRA on CNT's SPADE generator
LORA_RANK = 8
LORA_STEPS = 1500
LORA_LR = 1e-4
LORA_END_W = 1.0          # weight of the "keep t=0 reconstruction unchanged" term
N_TRAIN_PAIRS = 300       # training pairs: drawn from the 1,000-pair manifest, disjoint from the 120 eval pairs

SEED = 0
if MOCK: N_BANK, BANK_TOKENS_PER_IMG, LORA_STEPS, N_TRAIN_PAIRS = 20, 64, 30, 10   # tiny local dry run only

## 0. Install (once per session), then restart

In [ ]:
# ==== Install (once per session), then Runtime -> Restart session, then run "config" again and continue from "setup" ====
import os, sys, importlib.util, shutil, subprocess, glob
if not MOCK:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir("/content/package"):
        zp = PACKAGE_ZIP if os.path.exists(PACKAGE_ZIP) else next(iter(
            glob.glob(f"{MT}/textureMatcher_code_snapshot*.zip") + glob.glob(f"{DRIVE}/*/textureMatcher_code_snapshot*.zip")), None)
        assert zp, "textureMatcher_code_snapshot.zip not found -- set PACKAGE_ZIP in the config cell"
        print("package zip:", zp)
        os.system(f"rm -rf /content/package_unzipped; unzip -qo '{zp}' -d /content/package_unzipped")
        src = "/content/package_unzipped"; e = os.listdir(src)
        if len(e) == 1 and os.path.isdir(f"{src}/{e[0]}"): src = f"{src}/{e[0]}"
        shutil.copytree(src, "/content/package")
    os.system('pip install -q timm lpips einops gdown')
    os.system('cd /content && rm -rf cnt-siga24 && git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24 '
              '&& cd cnt-siga24 && git checkout -q 07269302610c92ce38e2dbbe31c492fd09e1c73e')
    os.system(f'rm -rf /content/cnt-siga24/ckpts; ln -sfn "{CNT_CKPTS}" /content/cnt-siga24/ckpts')
    CK = '/content/cnt-siga24/ckpts/finetuned-model256.ckpt'
    assert os.path.exists(CK), f'no finetuned-model256.ckpt in {CNT_CKPTS}'
    os.chdir('/content/cnt-siga24'); sys.path.insert(0, '/content/cnt-siga24')
    try:
        from src.inference.core import TexAutoEncoderInference, load_image
    except Exception as e:
        print('installing missing deps:', type(e).__name__)
        os.system('pip install -q -r requirements.in')
    if importlib.util.find_spec('detectron2') is None:   # build against the installed torch (~5-10 min)
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation",
                            "git+https://github.com/facebookresearch/detectron2.git"], capture_output=True, text=True)
        if r.returncode: print("detectron2 build FAILED:\n", r.stderr[-3000:])
    os.system('pip install -q -U timm "hydra-core>=1.3" "omegaconf>=2.3" "antlr4-python3-runtime==4.9.3"')
    os.system('bash /content/package/install/scoring.sh')
    os.system('pip install -q -U "diffusers>=0.39" transformers')                      # AutoencoderRAE (DINO decoder); keep hub/transformers in step
    os.system('SAM2_BUILD_CUDA=0 pip install -q "git+https://github.com/facebookresearch/sam2.git"')   # SAM2 encoder
    os.chdir('/content')
    for m in ("detectron2", "diffusers", "sam2", "lpips"):
        print(f"{m:>11}:", "OK" if importlib.util.find_spec(m) else "MISSING")
    print('>>> Now: Runtime -> Restart session, then run the "config" cell again and continue from "setup" down.')

## 1. Setup (after the restart: run config, then this)

In [ ]:
# ==== Setup (after the restart: run "config" first) ======================================================================
import os, sys, json, glob, math, time, random, shutil, subprocess
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.optimize import linear_sum_assignment
try:
    from IPython.display import display
except Exception:
    display = print
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
if not MOCK:
    from google.colab import drive
    drive.mount("/content/drive")
    for p in ("/content/package", "/content/cnt-siga24"):
        if p not in sys.path: sys.path.insert(0, p)
else:
    for p in (os.environ["CVPR_PACKAGE"], os.environ["CVPR_MOCKLIB"]):
        if p not in sys.path: sys.path.insert(0, p)
from texton_matching import matching, scoring, data as tmdata
MATCH_KW = {} if not MOCK else dict(n_sub=256, n_exact=128)     # the paper's defaults on Colab
for d in ("renders", "feats", "scores", "figs", "feature_level", "lora"):
    os.makedirs(f"{OUT}/{d}", exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def find_first(patterns):
    for p in patterns:
        hits = sorted(glob.glob(p))
        if hits: return hits[0]
    return None

# ---- data: DTD, the 1,000-pair manifest, the 120-pair subset (all as in the paper) ----
if not MOCK and len(glob.glob(f"{DTD_ROOT}/*/*.jpg")) < 5000:
    print("DTD not found at", DTD_ROOT, "-> downloading (~600 MB)")
    DTD_ROOT = str(tmdata.download_dtd(os.path.dirname(DTD_ROOT)))
MANIFEST = find_first([f"{MT}/tm_scale1000/manifest.json", f"{MT}/*/tm_scale1000/manifest.json"])
if MANIFEST is None:
    print("1,000-pair manifest not found under", MT, "-> rebuilding it (deterministic, seed 2026)")
    MANIFEST = str(tmdata.build_manifest_1000(DTD_ROOT, "/content/tm_scale1000"))
MAN = json.load(open(MANIFEST)); BYK = {int(p["k"]): p for p in MAN["pairs"]}
P120 = find_first([f"{MT}/pairs_120/pairs.csv", f"{MT}/*/pairs_120/pairs.csv", f"{MT}/pairs_for_*/pairs.csv",
                   f"{MT}/*/pairs_for_*/pairs.csv"])
if P120 is None:
    print("120-pair subset not found -> rebuilding it (deterministic, seed 2026)")
    P120 = str(tmdata.build_manifest_120(DTD_ROOT, MANIFEST, "/content/pairs_120"))
EVAL_KS = sorted(int(k) for k in pd.read_csv(P120)["k"])[:N_PAIRS]
assert all(k in BYK for k in EVAL_KS), "120-pair csv and 1,000-pair manifest disagree"
PAIRS = [BYK[k] for k in EVAL_KS]
REF_INC = find_first([f"{MT}/matching_ablation/ref_inc.npy", f"{MT}/*/matching_ablation/ref_inc.npy",
                      f"{OUT}/matching_ablation/ref_inc.npy"])
print(f"manifest: {MANIFEST}\n120 pairs: {P120} -> {len(PAIRS)} pairs\nKID reference: {REF_INC or 'not found (rebuilt in F)'}")

def endpoint(cat, fn):
    """The paper's preprocessing: 128 px endpoint (upsampled by each model's own loader)."""
    return tmdata.load128(cat, fn, DTD_ROOT)

# ---- render store: one .npz per (method, pair) with frames (len(T_GRID), 256, 256, 3) uint8 -> resumable ----
RDIR = f"{OUT}/renders"
def rpath(method, k): return f"{RDIR}/{method}/{int(k):04d}.npz"
def have(method, k): return os.path.exists(rpath(method, k))
def save_frames(method, k, frames):
    arr = np.stack([np.asarray(f.convert("RGB").resize((256, 256), Image.BICUBIC)) if isinstance(f, Image.Image)
                    else np.asarray(f) for f in frames]).astype(np.uint8)
    os.makedirs(f"{RDIR}/{method}", exist_ok=True)
    tmp = rpath(method, k)[:-4] + "_tmp.npz"
    np.savez_compressed(tmp, frames=arr, t=np.array(T_GRID[:len(arr)]))
    os.replace(tmp, rpath(method, k))
def load_frames(method, k): return np.load(rpath(method, k))["frames"]
def methods_on_disk(): return sorted(d for d in os.listdir(RDIR) if os.path.isdir(f"{RDIR}/{d}"))
def ks_on_disk(method): return sorted(int(f[:4]) for f in os.listdir(f"{RDIR}/{method}") if f.endswith(".npz") and "_tmp" not in f)
def to01(pil, size): return torch.from_numpy(np.asarray(pil.convert("RGB").resize((size, size), Image.BICUBIC)).copy()).float().permute(2, 0, 1)[None] / 255

# ---- the matching family on generic token sets (DINO / SAM / RAE). Same rules and definitions as the paper ----
def _nearest(x, y, bs=4096):
    return torch.cat([torch.cdist(x[i:i + bs], y).argmin(1) for i in range(0, len(x), bs)])

def _soft_bary(src, tgt, eps_rel=None, iters=100, eps_abs=None):
    """Balanced entropic OT (log domain, uniform masses), barycentric projection. eps relative to the median cost.
    Also returns the plan's effective number of partners per source token (exp of row entropy; 1 = one-to-one)."""
    C = torch.cdist(src, tgt).pow(2); eps = eps_abs if eps_abs is not None else eps_rel * C.median().clamp_min(1e-12)
    n, m = C.shape; la, lb = -math.log(n), -math.log(m)
    f = torch.zeros(n, device=src.device); g = torch.zeros(m, device=src.device)
    for _ in range(iters):
        f = eps * (la - torch.logsumexp((g[None] - C) / eps, 1))
        g = eps * (lb - torch.logsumexp((f[:, None] - C) / eps, 0))
    W = torch.softmax((f[:, None] + g[None] - C) / eps, 1)
    eff = float(torch.exp(-(W * torch.log(W.clamp_min(1e-30))).sum(1)).mean())
    return W @ tgt, eff

TOKEN_EFF = {}   # last effective-partners value per soft rule (diagnostic)

def cnt_field_sharp(cnt, d, seed, eps):
    """matching.field(rule='soft_ot') with the same seeded 4,096-sample draw and nearest-sample extension, but a log-domain
    solver: at eps = 0.005 the package's float32 multiplicative Sinkhorn underflows (exp(-C/eps) -> 0)."""
    L, C, H, W, sf, tf = cnt.grids(d)
    n_sub = MATCH_KW.get("n_sub", 4096)
    i_s = matching._sub(sf.shape[0], seed, "s", n_sub, sf.device); i_t = matching._sub(sf.shape[0], seed, "t", n_sub, sf.device)
    tt, eff = _soft_bary(sf[i_s], tf[i_t], eps_abs=eps)
    TOKEN_EFF["cnt_soft_ot_sharp"] = eff
    return dict(U=tt[_nearest(sf, sf[i_s])], X=sf, Y=tf, L=L, C=C, H=H, W=W)
def token_targets(Xa, Xb, rule, seed, n_sub=4096):
    """Target u_i for every token x_i of A; the path is (1-t) x_i + t u_i, on A's layout (as in the paper)."""
    Xa, Xb = Xa.float(), Xb.float(); N = len(Xa)
    if rule == "field_linear":                      # positional: token i of A blends with token i of B
        return Xb.clone()
    if rule == "target_mean":
        return Xb.mean(0, keepdim=True).expand_as(Xa).clone()
    if rule == "gaussian":
        v = 0.5 * float(Xa.var(0).mean() + Xb.var(0).mean())
        return matching._gaussian_map(Xa, Xb, ridge=GAUSS_RIDGE_REL * v).float()
    g = torch.Generator().manual_seed(int(seed) * 7919 + 1)
    if rule in ("soft_ot", "soft_ot_sharp"):
        i_s = torch.randperm(N, generator=g)[:n_sub].to(Xa.device); j_t = torch.randperm(len(Xb), generator=g)[:n_sub].to(Xa.device)
        tt, TOKEN_EFF[rule] = _soft_bary(Xa[i_s], Xb[j_t], EPS_REL if rule == "soft_ot" else EPS_REL_SHARP)
        return tt[_nearest(Xa, Xa[i_s])]
    if rule == "exact_ot":
        n = min(N, len(Xb), N_EXACT_TOK)
        i_s = torch.randperm(N, generator=g)[:n].to(Xa.device); j_t = torch.randperm(len(Xb), generator=g)[:n].to(Xa.device)
        r, c = linear_sum_assignment(torch.cdist(Xa[i_s], Xb[j_t]).pow(2).double().cpu().numpy())
        src, tt = Xa[i_s[torch.as_tensor(r, device=Xa.device)]], Xb[j_t[torch.as_tensor(c, device=Xa.device)]]
        return tt[_nearest(Xa, src)]
    raise ValueError(rule)

# ---- lazily loaded models (each section loads only what it needs) ----
_M = {}
def get_cnt():
    if "cnt" not in _M:
        if MOCK:
            import mock_env; _M["cnt"] = mock_env.MockCNT(DEV)
        else:
            from texton_matching.cnt import CNTModel
            _M["cnt"] = CNTModel("/content/cnt-siga24", device=DEV)
    return _M["cnt"]
def get_rae():
    if "rae" not in _M:
        if MOCK:
            import mock_env; _M["rae"] = mock_env.MockRAE().to(DEV).eval()
        else:
            from diffusers import AutoencoderRAE
            _M["rae"] = AutoencoderRAE.from_pretrained(RAE_REPO).to(DEV).eval()
    return _M["rae"]
def get_sam():
    if "sam" not in _M:
        if MOCK:
            import mock_env; _M["sam"] = mock_env.MockSAM().to(DEV).eval()
        else:
            from sam2.build_sam import build_sam2
            from huggingface_hub import hf_hub_download
            ck = hf_hub_download("facebook/sam2-hiera-small", "sam2_hiera_small.pt")
            try:
                _M["sam"] = build_sam2("sam2_hiera_s.yaml", ck, device=DEV).eval()
            except Exception:
                _M["sam"] = build_sam2("configs/sam2/sam2_hiera_s.yaml", ck, device=DEV).eval()
    return _M["sam"]
def get_lpips():
    if "lpips" not in _M:
        if MOCK:
            import mock_env; _M["lpips"] = mock_env.MockLPIPS()
        else:
            import lpips; _M["lpips"] = lpips.LPIPS(net="vgg", verbose=False).to(DEV).eval()
            for q in _M["lpips"].parameters(): q.requires_grad_(False)
    return _M["lpips"]

IMNET_M = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); IMNET_S = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
@torch.no_grad()
def enc_dino(pil):     # DINOv2-B tokens exactly as the RAE decoder consumes them: (256, 768)
    z = get_rae().encode(to01(pil, 224).to(DEV)).latent
    return z[0].flatten(1).t().contiguous()
@torch.no_grad()
def rae_decode(tokens):   # (B, 256, D) -> (B, 3, 256, 256) in [0, 1]
    B, N, D = tokens.shape; h = int(round(N ** 0.5))
    return get_rae().decode(tokens.transpose(1, 2).reshape(B, D, h, h)).sample.clamp(0, 1)
@torch.no_grad()
def enc_sam(pil):      # SAM2 image-encoder tokens: (4096, 256)
    x = ((to01(pil, 1024) - IMNET_M) / IMNET_S).to(DEV)
    out = get_sam().image_encoder(x)
    f = out["vision_features"] if isinstance(out, dict) else out
    return f[0].flatten(1).t().contiguous()
print("setup done | device", DEV)

## A. DINO + SAM at feature level (no images)

In [ ]:
# ==== A. DINO + SAM, feature level: does positional blending leave the space of real tokens? (no images) ==================
#   var_ret:   total token variance along the path / mean endpoint variance (1.0 = no contraction; < 1 = washed out).
#              This is the measure to read.
#   eff_partners (soft rules): how many B tokens each A token is effectively matched to (1 = one-to-one; hundreds = blur).
#   knn_ratio: distance to the nearest REAL token / same for the endpoints. Run 1 showed it is CONFOUNDED: blending pulls
#              tokens to the dense middle of the distribution, so collapse scores "more real". Kept for the record only.
# Rules: positional ("field_linear"), target_mean, gaussian, soft_ot, exact_ot -- same as the paper's family.
if RUN_FEATS:
    eval_files = {(p["cA"], p["fA"]) for p in PAIRS} | {(p["cB"], p["fB"]) for p in PAIRS}
    allf = sorted((os.path.basename(os.path.dirname(f)), os.path.basename(f)) for f in glob.glob(f"{DTD_ROOT}/*/*.jpg"))
    BANK_FILES = random.Random(7).sample([cf for cf in allf if cf not in eval_files], N_BANK)

    def build_bank(name, enc):
        path = f"{OUT}/feature_level/bank_{name}.pt"
        if os.path.exists(path): return torch.load(path).float().to(DEV)
        g = torch.Generator().manual_seed(3); toks = []
        for c, f in BANK_FILES:
            X = enc(endpoint(c, f))
            toks.append(X[torch.randperm(len(X), generator=g)[:BANK_TOKENS_PER_IMG].to(X.device)].half().cpu())
        bank = torch.cat(toks); torch.save(bank, path); return bank.float().to(DEV)

    def knn(Z, bank, bs=1024):
        return float(torch.cat([torch.cdist(Z[i:i + bs].float(), bank).min(1).values for i in range(0, len(Z), bs)]).mean())
    def totvar(Z): return float(((Z - Z.mean(0)) ** 2).sum(1).mean())

    for name, enc in [("dino", enc_dino), ("sam2", enc_sam)]:
        csv = f"{OUT}/feature_level/{name}.csv"
        old = pd.read_csv(csv) if os.path.exists(csv) else pd.DataFrame(columns=["k", "rule"])
        if len(old):   # rows from run 1 have no plan diagnostic: recompute just the soft-OT rows
            if "eff_partners" not in old: old["eff_partners"] = np.nan
            old = old[~(old.rule.str.startswith("soft") & old.eff_partners.isna())]
        done = set(zip(old["k"].astype(int), old["rule"])); rows = old.to_dict("records")
        t0 = time.time(); bank = build_bank(name, enc)
        print(f"[{name}] bank {tuple(bank.shape)} built/loaded in {time.time() - t0:.0f}s; {len(done)}/{len(PAIRS) * len(RULES)} (pair, rule) done")
        for i, p in enumerate(PAIRS):
            k = int(p["k"]); todo = [r for r in RULES if (k, r) not in done]
            if not todo: continue
            Xa, Xb = enc(endpoint(p["cA"], p["fA"])).float(), enc(endpoint(p["cB"], p["fB"])).float()
            kn0, v0 = 0.5 * (knn(Xa, bank) + knn(Xb, bank)), 0.5 * (totvar(Xa) + totvar(Xb))
            for rule in todo:
                U = token_targets(Xa, Xb, rule, k); eff = TOKEN_EFF.get(rule, np.nan) if rule.startswith("soft") else np.nan
                for t in T_GRID:
                    Z = (1 - t) * Xa + t * U
                    rows.append(dict(k=k, rule=rule, t=t, knn_ratio=knn(Z, bank) / kn0, var_ret=totvar(Z) / v0, eff_partners=eff))
            if (i + 1) % 10 == 0 or i == len(PAIRS) - 1:
                pd.DataFrame(rows).to_csv(csv, index=False)
                print(f"[{name}] {i + 1}/{len(PAIRS)} pairs ({(time.time() - t0) / 60:.1f} min)")
        df = pd.DataFrame(rows)
        if "eff_partners" not in df: df["eff_partners"] = np.nan
        mid = df[df.t == 0.5].groupby("rule")[["var_ret", "eff_partners", "knn_ratio"]].mean().reindex(RULES)
        print(f"\n[{name}] at t = 0.5 -- var_ret: 1 = no contraction | eff_partners: B tokens per A token (soft rules) | knn_ratio: confounded, ignore")
        display(mid.round(3))

## B1. DiffMorpher environment
The official code pins diffusers 0.17.1, which clashes with the RAE section, so it runs in its own Python 3.11 environment. SD 2.1-base was pulled from Hugging Face's official repo; the cell tries mirrors and downloads only safetensors and json files. The weights are cached in `masters_texture/cache/sd21_base` after the first download.

In [ ]:
# ==== B1. DiffMorpher environment: official code in its own Python 3.11 env (it pins diffusers 0.17.1), SD 2.1-base ====
DM_PY = "/content/dm_env/bin/python"
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode: raise RuntimeError(f"failed: {cmd}\n{(r.stdout + r.stderr)[-3000:]}")
    return r.stdout
def sd_weights_ok(d):
    return os.path.exists(f"{d}/model_index.json") and bool(glob.glob(f"{d}/unet/*.safetensors")) \
        and bool(glob.glob(f"{d}/text_encoder/*.safetensors")) and bool(glob.glob(f"{d}/vae/*.safetensors"))
if RUN_DIFFMORPHER and not MOCK:
    t0 = time.time()
    ok = os.path.exists(DM_PY) and subprocess.run([DM_PY, "-c", "import diffusers,torch;assert diffusers.__version__=='0.17.1' and torch.cuda.is_available()"]).returncode == 0
    if not ok:
        print("building the DiffMorpher env (~5 min) ...")
        sh("pip install -q uv")
        sh("uv venv /content/dm_env --python 3.11 --seed -q")
        sh(f'uv pip install -q --python {DM_PY} "torch==2.1.2+cu121" "torchvision==0.16.2+cu121" '
           f'--extra-index-url https://download.pytorch.org/whl/cu121 --index-strategy unsafe-best-match')
        sh(f"uv pip install -q --python {DM_PY} diffusers==0.17.1 transformers==4.34.1 accelerate==0.23.0 "
           f"huggingface_hub==0.19.4 safetensors==0.4.0 numpy==1.26.4 einops==0.7.0 lpips==0.1.4 tqdm Pillow packaging scipy")
    if not os.path.isdir("/content/DiffMorpher"):
        sh(f"git clone -q https://github.com/Kevin-thu/DiffMorpher.git /content/DiffMorpher && cd /content/DiffMorpher && git checkout -q {DM_COMMIT}")
    if not sd_weights_ok(SD_DIR):
        from huggingface_hub import snapshot_download
        pats = ["model_index.json", "scheduler/*", "tokenizer/*", "feature_extractor/*", "text_encoder/*.json",
                "unet/*.json", "vae/*.json", "text_encoder/*.safetensors", "unet/*.safetensors", "vae/*.safetensors"]
        for repo in SD_REPOS:     # safetensors + json only: no pickled weights from mirrors
            for ign in (["*fp16*", "*non_ema*", "*ema*"], []):
                try:
                    snapshot_download(repo, local_dir=SD_DIR, allow_patterns=pats, ignore_patterns=ign)
                except Exception as e:
                    print(f"  {repo}: {type(e).__name__}: {str(e)[:150]}"); break
                if sd_weights_ok(SD_DIR): break
            if sd_weights_ok(SD_DIR):
                print("SD 2.1-base weights from", repo); break
    assert sd_weights_ok(SD_DIR), f"could not get SD 2.1-base into {SD_DIR}; add a working mirror to SD_REPOS"
    DM_VARIANT = "" if glob.glob(f"{SD_DIR}/unet/diffusion_pytorch_model.safetensors") else "fp16"
    print(sh(f"{DM_PY} -c \"import torch,diffusers;print('dm env: torch',torch.__version__,'| diffusers',diffusers.__version__,'| cuda',torch.cuda.is_available())\"").strip())
    print(f"DiffMorpher ready in {(time.time() - t0) / 60:.1f} min | weights {SD_DIR} {('(variant ' + DM_VARIANT + ')') if DM_VARIANT else ''}")

## B2. Run DiffMorpher (resumable, time-boxed)

In [ ]:
# ==== B2. DiffMorpher on the 120 pairs (resumable; stops after DM_TIME_BUDGET_MIN) ========================================
# Official settings (LoRA 200 steps, AdaIN, self-attention lambda 0.6) except reschedule is OFF, so its 9 frames sit at
# exactly the T_GRID alphas and line up with every other method. Inputs are the same 128 px endpoints, upsampled to 512.
DM_DRIVER = r'''
import argparse, json, os, sys, time, shutil
import numpy as np, torch
from PIL import Image
ap = argparse.ArgumentParser()
for a in ("--pairs", "--out", "--model", "--dm_root", "--variant"): ap.add_argument(a, default="")
ap.add_argument("--budget_min", type=float); ap.add_argument("--lora_steps", type=int); ap.add_argument("--frames", type=int)
A = ap.parse_args()
sys.path.insert(0, A.dm_root); os.chdir(A.dm_root)
from model import DiffMorpherPipeline
from diffusers.models.attention_processor import AttnProcessor
kw = dict(torch_dtype=torch.float32)
if A.variant: kw["variant"] = A.variant
pipe = DiffMorpherPipeline.from_pretrained(A.model, **kw).to("cuda")
pairs = json.load(open(A.pairs)); t0 = time.time(); n_done = 0
os.makedirs(A.out, exist_ok=True)
for i, p in enumerate(pairs):
    out = f"{A.out}/{p['k']:04d}.npz"
    if os.path.exists(out): continue
    if (time.time() - t0) / 60 > A.budget_min:
        print(f"[dm] time budget reached after {n_done} pairs; rerun to continue", flush=True); break
    s = time.time(); tmp = f"/content/dm_tmp/{p['k']:04d}"; lora = "/content/dm_lora"
    shutil.rmtree(lora, ignore_errors=True); os.makedirs(lora); os.makedirs(tmp, exist_ok=True)
    pipe.unet.set_attn_processor(AttnProcessor())
    imgs = pipe(img_path_0=p["A"], img_path_1=p["B"], prompt_0=p["pA"], prompt_1=p["pB"], save_lora_dir=lora,
                lora_steps=A.lora_steps, use_adain=True, use_reschedule=False, lamd=0.6, num_frames=A.frames,
                output_path=tmp, save_intermediates=False)
    arr = np.stack([np.asarray(im.convert("RGB").resize((256, 256), Image.BICUBIC)) for im in imgs]).astype(np.uint8)
    assert len(arr) == A.frames, len(arr)
    np.savez_compressed(out[:-4] + "_tmp.npz", frames=arr, t=np.linspace(0, 1, A.frames)); os.replace(out[:-4] + "_tmp.npz", out)
    shutil.rmtree(tmp, ignore_errors=True); n_done += 1
    print(f"[dm] pair {p['k']:04d}: {time.time() - s:.0f}s  ({i + 1}/{len(pairs)}, {(time.time() - t0) / 60:.0f} min)", flush=True)
print("[dm] finished", flush=True)
'''
if RUN_DIFFMORPHER and not MOCK:
    todo = [p for p in PAIRS[:DM_N_PAIRS] if not have("diffmorpher", p["k"])]
    print(f"DiffMorpher: {DM_N_PAIRS - len(todo)}/{DM_N_PAIRS} pairs done, {len(todo)} to go")
    if todo:
        os.makedirs("/content/dm_inputs", exist_ok=True); jobs = []
        for p in todo:
            k = int(p["k"]); a, b = f"/content/dm_inputs/{k:04d}_A.png", f"/content/dm_inputs/{k:04d}_B.png"
            endpoint(p["cA"], p["fA"]).save(a); endpoint(p["cB"], p["fB"]).save(b)
            jobs.append(dict(k=k, A=a, B=b, pA=f"a photo of a {p['cA']} texture", pB=f"a photo of a {p['cB']} texture"))
        json.dump(jobs, open("/content/dm_pairs.json", "w"))
        open("/content/dm_driver.py", "w").write(DM_DRIVER)
        cmd = [DM_PY, "-u", "/content/dm_driver.py", "--pairs", "/content/dm_pairs.json", "--out", f"{RDIR}/diffmorpher",
               "--model", SD_DIR, "--dm_root", "/content/DiffMorpher", "--variant", DM_VARIANT,
               "--budget_min", str(DM_TIME_BUDGET_MIN), "--lora_steps", str(DM_LORA_STEPS), "--frames", str(len(T_GRID))]
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        tail = []
        for line in proc.stdout:
            tail = (tail + [line.rstrip()])[-40:]
            if line.startswith("[dm]"): print(line.rstrip(), flush=True)
        if proc.wait(): print("DiffMorpher driver FAILED; last output:\n" + "\n".join(tail))
    print(f"DiffMorpher pairs on disk: {len(ks_on_disk('diffmorpher')) if os.path.isdir(f'{RDIR}/diffmorpher') else 0}")

## C. CNT: every rule at every t

In [ ]:
# ==== C. CNT along the whole path: every rule at every t in T_GRID (the paper's operator, unchanged) =====================
if RUN_CNT_SWEEP:
    cnt = get_cnt()
    if "lora_mods" in globals(): set_lora(False)          # never render the base rules with the fine-tuned decoder
    t0 = time.time()
    for i, p in enumerate(PAIRS):
        k = int(p["k"]); todo = [r for r in RULES if not have(f"cnt_{r}", k)]
        if not todo: continue
        with torch.no_grad():
            d = cnt.build_pair(cnt.load_image(endpoint(p["cA"], p["fA"])), cnt.load_image(endpoint(p["cB"], p["fB"])))
            for rule in todo:
                if rule == "field_linear":
                    frames = [cnt.field_linear(d, t) for t in T_GRID]
                else:
                    if rule == "soft_ot_sharp": fd = cnt_field_sharp(cnt, d, k, CNT_EPS_SHARP)
                    else: fd = matching.field(cnt, d, rule, seed=k, **MATCH_KW)
                    frames = [cnt.decode_field(d, matching.zt(fd, t), fd["L"], fd["H"], fd["W"], fd["C"]) for t in T_GRID]
                    del fd
                save_frames(f"cnt_{rule}", k, [cnt.to_pil(f) for f in frames])
        if (i + 1) % 10 == 0:
            print(f"CNT sweep {i + 1}/{len(PAIRS)} ({(time.time() - t0) / 60:.1f} min) | sharp plan: "
                  f"{TOKEN_EFF.get('cnt_soft_ot_sharp', float('nan')):.1f} effective partners per cell")
    print(f"CNT sweep done ({(time.time() - t0) / 60:.1f} min)")

## D. LoRA-fine-tuned CNT decoder vs soft OT

In [ ]:
# ==== D. LoRA on CNT's generator: can fine-tuning make POSITIONAL blending as good as soft OT? ===========================
# Student: the frozen encoder + positional (field-linear) blend, decoded by the generator WITH LoRA adapters (rank LORA_RANK
# on every conv of the SPADE blocks). Teacher: the frozen generator rendering the paper's soft-OT field at the same t.
# Loss: LPIPS + L1 to the teacher at random t in [0.15, 0.85], plus a term that keeps the t = 0 reconstruction unchanged.
# Trained on N_TRAIN_PAIRS manifest pairs that are NOT in the 120 eval pairs; evaluated on the 120 as "cnt_lora_linear".
class LoRAConv2d(nn.Module):
    def __init__(self, base, r, alpha=None):
        super().__init__()
        self.base, self.scale, self.enabled = base, (alpha or r) / r, True
        self.down = nn.Conv2d(base.in_channels, r, base.kernel_size, base.stride, base.padding, base.dilation,
                              bias=False, padding_mode=base.padding_mode)
        self.up = nn.Conv2d(r, base.out_channels, 1, bias=False)
        nn.init.kaiming_uniform_(self.down.weight, a=math.sqrt(5)); nn.init.zeros_(self.up.weight)
    def forward(self, x):
        y = self.base(x)
        return y + self.scale * self.up(self.down(x)) if self.enabled else y

def inject_lora(module, r):
    mods = {}
    targets = [(n, m) for n, m in module.named_modules() if isinstance(m, nn.Conv2d) and m.groups == 1]
    for name, conv in targets:
        parent = module.get_submodule(name.rsplit(".", 1)[0]) if "." in name else module
        lm = LoRAConv2d(conv, r).to(conv.weight.device); setattr(parent, name.rsplit(".", 1)[-1], lm); mods[name] = lm
    for q in module.parameters(): q.requires_grad_(False)
    for lm in mods.values():
        for q in list(lm.down.parameters()) + list(lm.up.parameters()): q.requires_grad_(True)
    return mods
def set_lora(on):
    for lm in lora_mods.values(): lm.enabled = on

def decode_sm(cnt, blob_ref, sm):
    """CNTModel.decode_with_style_maps, line for line, but differentiable (for the LoRA student)."""
    G = cnt.model.generator
    with torch.no_grad():
        L = cnt.model.splat(blob_ref); si = L["layer_level_layouts"][-1][0]["score_img"]
        fms = [round(si.shape[-1] * r) for r in G.feature_map_size_ratios]; sp = cnt._pyr(si)
        x = sp[fms[0]][:, 1:].amax(dim=1)[:, None].repeat(1, G.n_embedding, 1, 1).clone()
    for i in range(len(fms)):
        x = G.spade_blocks[i](x, sm[fms[i]])
        if i == len(fms) - 1: break
        elif fms[i] != fms[i + 1]: x = F.interpolate(x, size=x.shape[-1] * 2, mode="bilinear", align_corners=False)
    return torch.tanh(x.unsqueeze(1)).flatten(1, 2)

if RUN_LORA:
    cnt = get_cnt(); lp = get_lpips()
    if "lora_mods" not in globals():
        lora_mods = inject_lora(cnt.model.generator.spade_blocks, LORA_RANK)
    n_par = sum(q.numel() for lm in lora_mods.values() for q in list(lm.down.parameters()) + list(lm.up.parameters()))
    print(f"LoRA: {len(lora_mods)} conv layers, {n_par / 1e6:.2f}M trainable parameters")
    LORA_PATH = f"{OUT}/lora/cnt_lora_r{LORA_RANK}_s{LORA_STEPS}.pt"
    p0 = PAIRS[0]
    with torch.no_grad():   # sanity: with LoRA off, the differentiable decoder must equal the package's field-linear
        d = cnt.build_pair(cnt.load_image(endpoint(p0["cA"], p0["fA"])), cnt.load_image(endpoint(p0["cB"], p0["fB"])))
        sa, sb = cnt.build_style_maps(d["bf_a"], d["si_a"]), cnt.build_style_maps(d["bf_b"], d["si_b"])
        set_lora(False)
        diff = (decode_sm(cnt, d["blob_a"], {s: 0.5 * sa[s] + 0.5 * sb[s] for s in sa}) - cnt.field_linear(d, 0.5)).abs().max().item()
    print(f"sanity: |differentiable decode - package field_linear| = {diff:.2e} (should be ~0)")
    if os.path.exists(LORA_PATH):
        st = torch.load(LORA_PATH, map_location=DEV)
        for n, lm in lora_mods.items(): lm.down.weight.data.copy_(st[n]["down"]); lm.up.weight.data.copy_(st[n]["up"])
        LOG = st.get("_log", []); print("loaded trained LoRA from", LORA_PATH)
    else:
        train_ks = random.Random(1).sample(sorted(set(BYK) - set(EVAL_KS)), N_TRAIN_PAIRS)
        params = [q for lm in lora_mods.values() for q in list(lm.down.parameters()) + list(lm.up.parameters())]
        opt = torch.optim.AdamW(params, lr=LORA_LR, weight_decay=0.0)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, LORA_STEPS)
        rng = random.Random(2); LOG = []; t0 = time.time()
        for step in range(LORA_STEPS):
            k = rng.choice(train_ks); p = BYK[k]; t = rng.uniform(0.15, 0.85)
            with torch.no_grad():
                d = cnt.build_pair(cnt.load_image(endpoint(p["cA"], p["fA"])), cnt.load_image(endpoint(p["cB"], p["fB"])))
                sa, sb = cnt.build_style_maps(d["bf_a"], d["si_a"]), cnt.build_style_maps(d["bf_b"], d["si_b"])
                set_lora(False)
                fd = matching.field(cnt, d, "soft_ot", seed=k, **MATCH_KW)
                teacher = cnt.decode_field(d, matching.zt(fd, t), fd["L"], fd["H"], fd["W"], fd["C"]); del fd
                rec0 = decode_sm(cnt, d["blob_a"], sa)
            set_lora(True)
            student = decode_sm(cnt, d["blob_a"], {s: (1 - t) * sa[s] + t * sb[s] for s in sa})
            student0 = decode_sm(cnt, d["blob_a"], sa)
            l_mid = lp(student, teacher).mean() + (student - teacher).abs().mean()
            l_end = lp(student0, rec0).mean() + (student0 - rec0).abs().mean()
            loss = l_mid + LORA_END_W * l_end
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
            LOG.append(dict(step=step, l_mid=l_mid.item(), l_end=l_end.item()))
            if (step + 1) % 100 == 0:
                lg = pd.DataFrame(LOG[-100:]).mean()
                print(f"step {step + 1}/{LORA_STEPS}  mid {lg.l_mid:.4f}  end {lg.l_end:.4f}  ({(time.time() - t0) / 60:.1f} min)")
        st = {n: dict(down=lm.down.weight.detach().cpu(), up=lm.up.weight.detach().cpu()) for n, lm in lora_mods.items()}
        st["_log"] = LOG; torch.save(st, LORA_PATH); print("saved", LORA_PATH)
    if LOG:
        lg = pd.DataFrame(LOG); plt.figure(figsize=(6, 3))
        plt.plot(lg.step, lg.l_mid.rolling(50, min_periods=1).mean(), label="mid-path (vs soft-OT teacher)")
        plt.plot(lg.step, lg.l_end.rolling(50, min_periods=1).mean(), label="t=0 (keep reconstruction)")
        plt.xlabel("step"); plt.ylabel("LPIPS + L1"); plt.legend(); plt.title("LoRA training"); plt.tight_layout()
        plt.savefig(f"{OUT}/figs/lora_training.png", dpi=120); plt.show()
    # render the fine-tuned positional path on the 120 eval pairs
    set_lora(True); t0 = time.time(); drift = []
    for i, p in enumerate(PAIRS):
        k = int(p["k"])
        if have("cnt_lora_linear", k): continue
        with torch.no_grad():
            d = cnt.build_pair(cnt.load_image(endpoint(p["cA"], p["fA"])), cnt.load_image(endpoint(p["cB"], p["fB"])))
            sa, sb = cnt.build_style_maps(d["bf_a"], d["si_a"]), cnt.build_style_maps(d["bf_b"], d["si_b"])
            frames = [decode_sm(cnt, d["blob_a"], {s: (1 - t) * sa[s] + t * sb[s] for s in sa}) for t in T_GRID]
            set_lora(False); base0 = decode_sm(cnt, d["blob_a"], sa); set_lora(True)
            drift.append(float(lp(frames[0], base0).mean()))
        save_frames("cnt_lora_linear", k, [cnt.to_pil(f) for f in frames])
    set_lora(False)        # leave the generator in its original state for everything else
    if drift: print(f"LoRA renders done ({(time.time() - t0) / 60:.1f} min); t=0 drift vs the original decoder: LPIPS {np.mean(drift):.4f}")

## E. DINO at image level (RAE decoder)

In [ ]:
# ==== E. DINO at image level: interpolate DINOv2 tokens, decode with the RAE pixel decoder ================================
# Same rules as everywhere else, on the 256 DINOv2-B tokens of each endpoint (16 x 16 grid). The path stays on A's token
# layout, exactly like CNT's field. t = 0 is RAE's reconstruction of A (its own ceiling).
if RUN_RAE:
    t0 = time.time()
    for i, p in enumerate(PAIRS):
        k = int(p["k"]); todo = [r for r in RULES if not have(f"rae_{r}", k)]
        if not todo: continue
        Xa, Xb = enc_dino(endpoint(p["cA"], p["fA"])).float(), enc_dino(endpoint(p["cB"], p["fB"])).float()
        for rule in todo:
            U = token_targets(Xa, Xb, rule, k)
            Z = torch.stack([(1 - t) * Xa + t * U for t in T_GRID])
            imgs = rae_decode(Z.to(DEV))
            save_frames(f"rae_{rule}", k, [(im.permute(1, 2, 0).cpu().numpy() * 255).round().astype(np.uint8) for im in imgs])
        if (i + 1) % 20 == 0: print(f"RAE {i + 1}/{len(PAIRS)} ({(time.time() - t0) / 60:.1f} min)")
    p0 = PAIRS[0]; fr = load_frames("rae_field_linear", p0["k"])
    fig, ax = plt.subplots(1, 3, figsize=(8, 3))
    for a, im, ttl in zip(ax, [np.asarray(endpoint(p0["cA"], p0["fA"]).resize((256, 256))), fr[0], fr[-1]],
                          ["A (input)", "RAE recon of A", "positional t=1"]):
        a.imshow(im); a.set_title(ttl, fontsize=9); a.axis("off")
    plt.suptitle("RAE sanity: the t=0 frame should look like A"); plt.tight_layout(); plt.show()
    print(f"RAE done ({(time.time() - t0) / 60:.1f} min)")

## F. KID / FID along the path, Delta-KID, graphs and strips

In [ ]:
# ==== F. Scores along the path: KID / FID at every t, Delta-KID with paired jackknife CIs, graphs, strips =================
# Canonical protocol of the paper: Inception features, the frozen 2,819-image DTD reference, unbiased cubic-kernel KID,
# paired leave-one-pair-out jackknife. FID on 120 images is strongly biased upward (reported, but read KID).
if RUN_SCORE:
    if MOCK:
        import mock_env; ext = mock_env.MockExtractor()
    else:
        sys.path.insert(0, "/content/package"); import rescore; ext = rescore.Extractor(DEV)
    if REF_INC is None:
        from pathlib import Path
        ref_inc, _ = rescore.build_or_load_reference(Path(OUT), Path(DTD_ROOT), ext)
    else:
        ref_inc = np.load(REF_INC)
    kyy = scoring._reference_self_term(ref_inc)
    MU_R, COV_R = ref_inc.mean(0).astype(np.float64), np.cov(ref_inc.astype(np.float64), rowvar=False)
    def fid(gen):
        """Standard Frechet distance, exact and fast: with n < d, tr sqrt(S_g S_r) = sum sqrt(eig(Xc S_r Xc^T / (n-1)))."""
        g = gen.astype(np.float64); mu = g.mean(0); Xc = g - mu; n = len(g)
        lam = np.clip(np.linalg.eigvalsh(Xc @ COV_R @ Xc.T / (n - 1)), 0, None)
        return float(((mu - MU_R) ** 2).sum() + (Xc ** 2).sum() / (n - 1) + np.trace(COV_R) - 2 * np.sqrt(lam).sum())

    DINO_FE = {}
    def method_feats(m):
        ks = ks_on_disk(m); path = f"{OUT}/feats/{m}.npz"
        if os.path.exists(path):
            z = np.load(path)
            if list(z["ks"]) == ks and "dino" in z.files:
                DINO_FE[m] = z["dino"]; return ks, z["inc"]
        inc, dino = None, None
        for j0 in range(0, len(ks), 30):
            fr = np.stack([load_frames(m, k) for k in ks[j0:j0 + 30]])          # (b, T, H, W, 3)
            for ti in range(len(T_GRID)):
                pils = [Image.fromarray(x) for x in fr[:, ti]]
                fi, fd_ = ext.inc_feats(pils), ext.dino_feats(pils)
                if inc is None:
                    inc = np.zeros((len(T_GRID), len(ks), fi.shape[1]), np.float32); dino = np.zeros((len(T_GRID), len(ks), fd_.shape[1]), np.float32)
                inc[ti, j0:j0 + len(fr)], dino[ti, j0:j0 + len(fr)] = fi, fd_
        np.savez(path, ks=np.array(ks), inc=inc, dino=dino); DINO_FE[m] = dino; return ks, inc

    METHODS = [m for m in methods_on_disk() if ks_on_disk(m)]
    FE = {m: method_feats(m) for m in METHODS}
    # the original endpoints (same anchors for every method) in DINOv2 space, for the progress measure
    ep_path = f"{OUT}/feats/endpoints.npz"
    if os.path.exists(ep_path) and list(np.load(ep_path)["ks"]) == EVAL_KS:
        z = np.load(ep_path); EP_A, EP_B = z["a"], z["b"]
    else:
        EP_A = ext.dino_feats([endpoint(p["cA"], p["fA"]).resize((256, 256)) for p in PAIRS])
        EP_B = ext.dino_feats([endpoint(p["cB"], p["fB"]).resize((256, 256)) for p in PAIRS])
        np.savez(ep_path, ks=np.array(EVAL_KS), a=EP_A, b=EP_B)
    EIDX = {k: i for i, k in enumerate(EVAL_KS)}
    rows = []
    for m, (ks, inc) in FE.items():
        for ti, t in enumerate(T_GRID):
            rows.append(dict(method=m, t=t, n=len(ks), KID=scoring.kid_full(inc[ti], ref_inc, kyy), FID=fid(inc[ti])))
    S = pd.DataFrame(rows); S.to_csv(f"{OUT}/scores/kid_fid_by_t.csv", index=False)
    print("KID x 1000 by t (lower is better):"); display((S.pivot(index="method", columns="t", values="KID") * 1000).round(2))
    print("FID by t (n=120: biased upward, compare only within a column):"); display(S.pivot(index="method", columns="t", values="FID").round(1))
    # excess KID: each method minus its OWN t = 0 KID, so autoencoder quality (CNT vs RAE vs SD) does not decide the ranking
    S["KID_excess"] = S.KID - S.groupby("method").KID.transform(lambda v: v.iloc[0])
    print("Excess KID x 1000 over each method's own reconstruction (t = 0); this isolates the interpolation itself:")
    display((S.pivot(index="method", columns="t", values="KID_excess") * 1000).round(2))

    # progress: where each frame sits between the ORIGINAL A and B in DINOv2 space, place = dA / (dA + dB).
    # KID only says frames look real; a path that sits at A and snaps to B at the end can score well. This catches it.
    P = []
    for m, (ks, _) in FE.items():
        ia = np.array([EIDX[k] for k in ks]); dn = DINO_FE[m]
        for ti, t in enumerate(T_GRID):
            dA = np.linalg.norm(dn[ti] - EP_A[ia], axis=1); dB = np.linalg.norm(dn[ti] - EP_B[ia], axis=1)
            for k, pl in zip(ks, dA / (dA + dB + 1e-12)): P.append(dict(method=m, k=k, t=t, place=pl))
    P = pd.DataFrame(P); P.to_csv(f"{OUT}/scores/progress_by_t.csv", index=False)
    def prog_summary(g):
        w = g.pivot(index="k", columns="t", values="place").sort_index(axis=1).values      # (pairs, T)
        steps = np.diff(w, axis=1); span = w[:, -1] - w[:, 0]; moving = span > 0.05      # pairs whose path really moves
        return pd.Series(dict(place_t0=w[:, 0].mean(), place_mid=w[:, len(T_GRID) // 2].mean(), place_t1=w[:, -1].mean(),
                              span=span.mean(), balance_mid=np.abs(w[:, len(T_GRID) // 2] - 0.5).mean(),
                              monotone=(steps >= -0.01).mean(),
                              max_jump_share=np.median(steps[moving].max(1) / span[moving]) if moving.any() else np.nan))
    PS = P.groupby("method").apply(prog_summary).round(3); PS.to_csv(f"{OUT}/scores/progress_summary.csv")
    print("Progress along the path (DINOv2 place between the original A and B; 0 = at A, 1 = at B):")
    print("  balance_mid = |place - 0.5| at t = 0.5 (lower = more in between); monotone = share of steps that do not go back;")
    print("  span = how far the path travels from t=0 to t=1 (matching rules end on A's layout, so below 1 is expected);")
    print(f"  max_jump_share = median over moving pairs of biggest step / span (uniform path ~{1 / (len(T_GRID) - 1):.2f}; near 1 = it snaps)")
    display(PS)

    def delta(m, base):   # paired jackknife on the pairs both methods have
        (ka, fa), (kb, fb) = FE[base], FE[m]; common = sorted(set(ka) & set(kb))
        ia, ib = [ka.index(k) for k in common], [kb.index(k) for k in common]
        out = []
        for ti, t in enumerate(T_GRID):
            r = scoring.paired_jackknife_delta_kid(fb[ti][ib], fa[ti][ia], ref_inc)
            out.append(dict(method=m, base=base, t=t, n=len(common), dKID=r.delta, lo=r.lo, hi=r.hi, excl0=r.excludes_zero))
        return out
    COMPARE = [(m, "cnt_field_linear") for m in METHODS if m.startswith("cnt_") and m != "cnt_field_linear"]
    COMPARE += [(m, "rae_field_linear") for m in METHODS if m.startswith("rae_") and m != "rae_field_linear"]
    COMPARE += [("cnt_lora_linear", "cnt_soft_ot"), ("diffmorpher", "cnt_field_linear"), ("diffmorpher", "cnt_soft_ot")]
    D = pd.DataFrame([row for m, b in COMPARE if m in FE and b in FE for row in delta(m, b)])
    D.to_csv(f"{OUT}/scores/delta_kid_by_t.csv", index=False)
    if len(D):
        print("Delta-KID x 1000 vs base at t = 0.5 (negative = better than base; CI from paired jackknife):")
        mid = D[D.t == 0.5].copy()
        for c in ("dKID", "lo", "hi"): mid[c] = (mid[c] * 1000).round(2)
        display(mid[["method", "base", "n", "dKID", "lo", "hi", "excl0"]])

    # ---- graphs: KID / FID along the path ----
    COL = {"field_linear": "tab:blue", "target_mean": "tab:olive", "gaussian": "tab:purple", "soft_ot": "tab:green", "soft_ot_sharp": "tab:cyan", "exact_ot": "tab:red"}
    def style(m):
        rule = m.split("_", 1)[1] if "_" in m else m
        if m == "diffmorpher": return dict(color="black", ls="-", marker="s", lw=2, label="DiffMorpher")
        if m == "cnt_lora_linear": return dict(color="tab:blue", ls=":", marker="^", lw=2, label="CNT positional + LoRA")
        fam = "CNT" if m.startswith("cnt_") else "DINO-RAE"
        return dict(color=COL.get(rule, "gray"), ls="-" if fam == "CNT" else "--", marker="o", lw=1.6,
                    label=f"{fam} {rule.replace('field_linear', 'positional')}")
    for metric in ("KID", "FID"):
        fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=False)
        for ax, fam in zip(axes, ("cnt_", "rae_")):
            for m in METHODS:
                if m.startswith(fam) or m == "diffmorpher":
                    s = S[S.method == m]; ax.plot(s.t, s[metric] * (1000 if metric == "KID" else 1), **style(m))
            ax.set_xlabel("t (0 = A, 1 = B)"); ax.set_ylabel(metric + (" x 1000" if metric == "KID" else ""))
            ax.set_title(("CNT (the paper's representation)" if fam == "cnt_" else "DINOv2 tokens + RAE decoder") + f": {metric} along the path")
            ax.grid(alpha=.3); ax.legend(fontsize=8)
        plt.tight_layout(); plt.savefig(f"{OUT}/figs/{metric.lower()}_vs_t.png", dpi=140); plt.show()
    fig, axes = plt.subplots(1, 3, figsize=(20, 4.5))
    for ax, fam in zip(axes[:2], ("cnt_", "rae_")):
        for m in METHODS:
            if m.startswith(fam) or m == "diffmorpher":
                s = S[S.method == m]; ax.plot(s.t, s.KID_excess * 1000, **style(m))
        ax.axhline(0, color="k", lw=.8); ax.grid(alpha=.3); ax.legend(fontsize=7); ax.set_xlabel("t")
        ax.set_ylabel("KID - own KID at t=0 (x 1000)"); ax.set_title(("CNT" if fam == "cnt_" else "DINO-RAE") + ": excess KID over own reconstruction")
    ax = axes[2]
    for m in METHODS:
        g = P[P.method == m].groupby("t").place; mu, se = g.mean(), 1.96 * g.sem()
        st = style(m); ax.plot(mu.index, mu, **st); ax.fill_between(mu.index, mu - se, mu + se, color=st["color"], alpha=.08)
    ax.plot([0, 1], [0, 1], "k:", lw=1, label="uniform progress")
    ax.set_xlabel("t"); ax.set_ylabel("place between A and B (DINOv2)"); ax.set_title("Does the path actually move? (progress)")
    ax.grid(alpha=.3); ax.legend(fontsize=6)
    plt.tight_layout(); plt.savefig(f"{OUT}/figs/excess_kid_and_progress.png", dpi=140); plt.show()
    if len(D):
        fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
        for ax, base in zip(axes, ("cnt_field_linear", "rae_field_linear")):
            for m in D[D.base == base].method.unique():
                s = D[(D.base == base) & (D.method == m)]
                st = style(m); ax.plot(s.t, s.dKID * 1000, **st)
                ax.fill_between(s.t, s.lo * 1000, s.hi * 1000, color=st["color"], alpha=.12)
            ax.axhline(0, color="k", lw=.8); ax.grid(alpha=.3); ax.legend(fontsize=8)
            ax.set_xlabel("t"); ax.set_ylabel("Delta-KID x 1000 (negative = better)"); ax.set_title(f"vs {base.replace('field_linear', 'positional')} (95% jackknife CI)")
        plt.tight_layout(); plt.savefig(f"{OUT}/figs/delta_kid_vs_t.png", dpi=140); plt.show()
    print("\nBest interior t per method (lowest KID among 0 < t < 1):")
    inner = S[(S.t > 0) & (S.t < 1)]
    display(inner.loc[inner.groupby("method").KID.idxmin(), ["method", "t", "KID"]].assign(KID=lambda x: (x.KID * 1000).round(2)))

    # ---- strips: 3 pairs, every method, every t ----
    common = sorted(set.intersection(*[set(FE[m][0]) for m in METHODS])) if METHODS else []
    for k in random.Random(5).sample(common, min(3, len(common))):
        p = BYK[k]; fig, ax = plt.subplots(len(METHODS), len(T_GRID) + 2, figsize=(1.6 * (len(T_GRID) + 2), 1.7 * len(METHODS)), squeeze=False)
        A_, B_ = [np.asarray(endpoint(c, f).resize((256, 256))) for c, f in ((p["cA"], p["fA"]), (p["cB"], p["fB"]))]
        for r, m in enumerate(METHODS):
            fr = load_frames(m, k)
            for c, im in enumerate([A_] + list(fr) + [B_]):
                ax[r][c].imshow(im); ax[r][c].set_xticks([]); ax[r][c].set_yticks([])
                if r == 0: ax[r][c].set_title("A" if c == 0 else ("B" if c == len(T_GRID) + 1 else f"t={T_GRID[c - 1]:g}"), fontsize=8)
            ax[r][0].set_ylabel(style(m)["label"], fontsize=7, rotation=0, ha="right", va="center")
        plt.suptitle(f"pair {k:04d}: {p['cA']} -> {p['cB']}"); plt.tight_layout()
        plt.savefig(f"{OUT}/figs/strip_{k:04d}.png", dpi=110); plt.show()

    # ---- feature-level (section A) graphs ----
    for name in ("dino", "sam2"):
        csv = f"{OUT}/feature_level/{name}.csv"
        if not os.path.exists(csv): continue
        df = pd.read_csv(csv); fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        for ax, col, ttl in zip(axes, ("knn_ratio", "var_ret"), ("distance to nearest real token / endpoint value (1 = real)",
                                                                "variance retained (1 = none lost)")):
            for rule in RULES:
                g = df[df.rule == rule].groupby("t")[col]; mu, se = g.mean(), g.sem() * 1.96
                ax.plot(mu.index, mu, marker="o", color=COL[rule], label=rule.replace("field_linear", "positional"))
                ax.fill_between(mu.index, mu - se, mu + se, color=COL[rule], alpha=.15)
            ax.axhline(1, color="k", lw=.8); ax.set_xlabel("t"); ax.set_title(f"{name.upper()}: {ttl}", fontsize=9); ax.grid(alpha=.3); ax.legend(fontsize=8)
        plt.tight_layout(); plt.savefig(f"{OUT}/figs/feature_level_{name}.png", dpi=130); plt.show()
    print("all figures in", f"{OUT}/figs", "| tables in", f"{OUT}/scores")

## Reading the results
- **`figs/kid_vs_t.png`:** one line per method along the whole path. The left panel is CNT plus LoRA and DiffMorpher; the right panel is DINO-RAE plus DiffMorpher. "Best interior t" in the table tells you if some t beats 0.5.
- **`figs/excess_kid_and_progress.png`:** two checks KID alone can't do.
  - **Left two panels:** KID minus each method's *own* t = 0 KID, so the comparison is about interpolation, not about which autoencoder reconstructs best. This matters most for DiffMorpher vs CNT.
  - **Right panel:** progress, i.e. where each frame sits between the real A and B in DINOv2 space. A good path follows the diagonal. A curve that stays flat and jumps at the end is a snap, even when its KID looks great. The table gives `balance_mid` (how in-between t = 0.5 is) and `max_jump_share` (about 0.12 for a uniform path, near 1 for a snap).
- **`figs/delta_kid_vs_t.png`:** each method minus positional, with 95% CIs. Below zero means better than positional. A band that excludes zero across most of the path is the headline.
- **D:** if `cnt_lora_linear` reaches soft OT, matching gives for free what otherwise needs training. If it doesn't, positional blending can't be trained into it. Report it either way.
- **A:** positional with `knn_ratio` above 1 and matching near 1 means positional leaves the space of real tokens. This is the non-circular DINO and SAM evidence.